In [0]:
# Input and output locations for the PDF parsing notebook.
respath = "development_042_silver_sandbox.demand_forecast"
pdf_path = "/Volumes/development_042_silver_sandbox/evan_dev/games_white_papers/"

# Process this many PDFs. Use -1 to process all PDFs.
LIMIT = -1

# The parser notebook will write its page-level handoff dataset here.
OUTPUT_TABLE = f"{respath}.pdf_pages_for_maverick_v1"

# Basic configuration checks.
if LIMIT != -1 and LIMIT < 1:
    raise ValueError("LIMIT must be -1 (all PDFs) or a positive number.")

if not pdf_path.endswith("/"):
    pdf_path += "/"

print(f"PDF folder: {pdf_path}")
print(f"PDF limit: {LIMIT}")
print(f"Output table: {OUTPUT_TABLE}")

In [0]:
def list_pdf_files(root_path):
    """Return PDF file metadata from a folder and its subfolders."""
    pending_paths = [root_path]
    pdf_files = []

    while pending_paths:
        current_path = pending_paths.pop()
        for item in dbutils.fs.ls(current_path):
            if item.isDir():
                pending_paths.append(item.path)
            elif item.name.lower().endswith(".pdf"):
                pdf_files.append({
                    "file_name": item.name,
                    "source_path": item.path,
                    "file_size_bytes": item.size,
                })

    return pdf_files


def select_pdf_files(pdf_files, limit):
    """Sort PDFs by path and apply the requested limit."""
    sorted_files = sorted(
        pdf_files,
        key=lambda item: (item["source_path"].lower(), item["source_path"])
    )

    if limit == -1:
        return sorted_files

    return sorted_files[:limit]


all_pdf_files = list_pdf_files(pdf_path)
selected_pdf_files = select_pdf_files(all_pdf_files, LIMIT)

print(f"PDFs found: {len(all_pdf_files)}")
print(f"PDFs selected: {len(selected_pdf_files)}")

if selected_pdf_files:
    display(spark.createDataFrame(selected_pdf_files))
else:
    print("No PDF files found. Check pdf_path.")

In [0]:
import hashlib
import re

# Root folder for all rendered page images.
IMAGE_OUTPUT_ROOT = f"{pdf_path.rstrip('/')}/_parsed_page_images/"
def get_game_name(pdf_file):
    """Derive the game name from the PDF file name."""
    name = pdf_file["file_name"]
    if name.lower().endswith(".pdf"):
        name = name[:-4]
    return name.strip().lower()

def make_image_folder_name(pdf_file):
    """Build a filesystem-safe, unique folder name from a PDF filename and path."""
    file_name = pdf_file["file_name"]
    stem = re.sub(r"(?i)\.pdf$", "", file_name)

    safe_stem = re.sub(r"[^A-Za-z0-9_-]+", "_", stem)
    safe_stem = safe_stem.strip("_").lower() or "game"

    # Include a short path hash to avoid collisions between same-named PDFs.
    path_hash = hashlib.sha1(
        pdf_file["source_path"].encode("utf-8")
    ).hexdigest()[:8]

    return f"{safe_stem}_{path_hash}"


def prepare_image_folders(pdf_files, image_root):
    """Create and return a dedicated rendered-image folder for each PDF."""
    dbutils.fs.mkdirs(image_root)
    folders_by_source_path = {}

    for pdf_file in pdf_files:
        folder_name = make_image_folder_name(pdf_file)
        image_folder = f"{image_root}{folder_name}/"
        dbutils.fs.mkdirs(image_folder)
        folders_by_source_path[pdf_file["source_path"]] = image_folder

    return folders_by_source_path


IMAGE_FOLDER_BY_PATH = prepare_image_folders(
    selected_pdf_files,
    IMAGE_OUTPUT_ROOT,
)

print(f"Image output root: {IMAGE_OUTPUT_ROOT}")
print(f"Per-PDF image folders prepared: {len(IMAGE_FOLDER_BY_PATH)}")

if IMAGE_FOLDER_BY_PATH:
    display(spark.createDataFrame([
        {
            "source_path": source_path,
            "image_output_folder": image_folder,
        }
        for source_path, image_folder in IMAGE_FOLDER_BY_PATH.items()
    ]))

In [0]:
from pyspark.sql import functions as F

PARSED_OUTPUT_TABLE = f"{respath}.pdf_parse_results_v1"

# The parser options must be literal for a Spark batch.
# All rendered page images will be saved under this shared folder.
dbutils.fs.mkdirs(IMAGE_OUTPUT_ROOT)

PARSE_OPTIONS = {
    "version": "2.0",
    "imageOutputPath": IMAGE_OUTPUT_ROOT,
    "descriptionElementTypes": "",
}


def get_game_name(pdf_file):
    """Derive the game name from the PDF file name."""
    name = pdf_file["file_name"]
    if name.lower().endswith(".pdf"):
        name = name[:-4]
    return name.strip().lower()


def build_batch_metadata(pdf_files):
    """Prepare one metadata row for each selected PDF."""
    return [
        {
            "game_name": get_game_name(pdf_file),
            "file_name": pdf_file["file_name"],
            "source_path": pdf_file["source_path"],
            "file_size_bytes": int(pdf_file["file_size_bytes"]),
            "image_output_folder": IMAGE_OUTPUT_ROOT,
        }
        for pdf_file in pdf_files
    ]


def parse_selected_pdfs_in_batch(pdf_files):
    """Parse the selected PDFs together in a distributed Spark operation."""
    if not pdf_files:
        raise ValueError("No PDFs are selected. Check Cell 2 and pdf_path.")

    metadata_df = spark.createDataFrame(build_batch_metadata(pdf_files))
    selected_paths = [pdf_file["source_path"] for pdf_file in pdf_files]

    # Read the selected files together and match each one to its metadata.
    binary_df = (
        spark.read
        .format("binaryFile")
        .load(selected_paths)
        .select(
            F.col("path").alias("binary_path"),
            F.col("content"),
        )
    )

    files_to_parse = (
        binary_df.alias("b")
        .join(
            F.broadcast(metadata_df.alias("m")),
            F.col("b.binary_path") == F.col("m.source_path"),
            "inner",
        )
        .select(
            F.col("m.game_name"),
            F.col("m.file_name"),
            F.col("m.source_path"),
            F.col("m.file_size_bytes"),
            F.col("m.image_output_folder"),
            F.col("b.content"),
        )
    )

    # Spark can distribute these rows across available partitions.
    parsed = files_to_parse.withColumn(
        "parsed_result",
        F.ai_parse_document(
            F.col("content"),
            options=PARSE_OPTIONS,
        ),
    )

    return (
        parsed
        .withColumn("parsed_json", F.to_json("parsed_result"))
        .withColumn(
            "parse_status",
            F.when(
                F.expr("parsed_result:document IS NOT NULL"),
                F.lit("success"),
            ).otherwise(F.lit("failed")),
        )
        .withColumn(
            "error_message",
            F.when(
                F.col("parse_status") == "failed",
                F.coalesce(
                    F.expr(
                        "parsed_result:error_status[0].error_message::STRING"
                    ),
                    F.lit("Parser returned no document; inspect parsed_json."),
                ),
            ),
        )
        .select(
            "game_name",
            "file_name",
            "source_path",
            "file_size_bytes",
            "image_output_folder",
            "parsed_json",
            "parse_status",
            "error_message",
        )
    )


if not selected_pdf_files:
    raise ValueError("No PDFs are selected. Check Cell 2 and pdf_path.")

parsed_results_df = parse_selected_pdfs_in_batch(selected_pdf_files)

(
    parsed_results_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PARSED_OUTPUT_TABLE)
)

print(f"Saved parse results to: {PARSED_OUTPUT_TABLE}")

display(
    spark.table(PARSED_OUTPUT_TABLE)
    .select(
        "game_name",
        "parse_status",
        "image_output_folder",
        "error_message",
    )
    .orderBy("game_name")
)

In [0]:
from pyspark.sql.types import (
    ArrayType,
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

# The fields needed from ai_parse_document's version 2.0 output.
bbox_schema = ArrayType(StructType([
    StructField("coord", ArrayType(IntegerType()), True),
    StructField("page_id", IntegerType(), True),
]))

element_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("type", StringType(), True),
    StructField("content", StringType(), True),
    StructField("confidence", DoubleType(), True),
    StructField("bbox", bbox_schema, True),
    StructField("description", StringType(), True),
])

parsed_document_schema = StructType([
    StructField("document", StructType([
        StructField("pages", ArrayType(StructType([
            StructField("id", IntegerType(), True),
            StructField("image_uri", StringType(), True),
        ])), True),
        StructField("elements", ArrayType(element_schema), True),
    ]), True),
    StructField("error_status", ArrayType(StructType([
        StructField("error_message", StringType(), True),
        StructField("page_id", IntegerType(), True),
    ])), True),
    StructField("metadata", StructType([
        StructField("id", StringType(), True),
        StructField("version", StringType(), True),
    ]), True),
])


def build_page_handoff(parsed_table, output_table):
    """Flatten successful parser results into one row per PDF page."""
    parsed = (
        spark.table(parsed_table)
        .filter(F.col("parse_status") == "success")
        .withColumn(
            "parsed",
            F.from_json("parsed_json", parsed_document_schema),
        )
        .withColumn(
            "page",
            F.explode(F.col("parsed.document.pages")),
        )
        # Associate each extracted element with the page referenced by its bbox.
        .withColumn(
            "page_elements",
            F.expr("""
                filter(
                    parsed.document.elements,
                    e -> exists(e.bbox, b -> b.page_id = page.id)
                )
            """),
        )
        # Keep readable text and the full structured elements for downstream use.
        .withColumn(
            "page_text",
            F.expr("""
                concat_ws(
                    '\\n',
                    transform(
                        filter(
                            page_elements,
                            e -> e.content IS NOT NULL
                                 AND trim(e.content) <> ''
                        ),
                        e -> e.content
                    )
                )
            """),
        )
        .select(
            "game_name",
            "file_name",
            "source_path",
            (F.col("page.id")).alias("page_index"),
            (F.col("page.id") + F.lit(1)).alias("page_number"),
            F.col("page.image_uri").alias("page_image_path"),
            "page_text",
            F.to_json("page_elements").alias("page_elements_json"),
            F.size("page_elements").alias("page_element_count"),
            F.col("parsed.metadata.version").alias("parser_version"),
            F.to_json("parsed.error_status").alias("parser_errors_json"),
            "image_output_folder",
        )
    )

    (
        parsed.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(output_table)
    )

    return spark.table(output_table)


page_handoff_df = build_page_handoff(
    PARSED_OUTPUT_TABLE,
    OUTPUT_TABLE,
)

print(f"Page-level handoff table: {OUTPUT_TABLE}")
print(f"Page rows: {page_handoff_df.count()}")
display(page_handoff_df.orderBy("game_name", "page_number"))